# 02: Multi-Head Self-Attention & Transformer Encoders from Scratch

**Track 10: Deep Sequential Models & Transformers from Scratch** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Implement the Transformer architecture from scratch in PyTorch: Scaled Dot-Product Attention, Multi-Head projections, Sinusoidal Positional Encoding, and LayerNorm residual blocks.


## 1. Scaled Dot-Product Attention Formulation
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

In [ ]:
import torch
import torch.nn as nn
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.scale = 1.0 / math.sqrt(d_k)
        
    def forward(self, q, k, v, mask=None):
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)
        return output, attn_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.attention = ScaledDotProductAttention(self.d_k)
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        out, weights = self.attention(q, k, v, mask=mask)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.num_heads * self.d_k)
        return self.w_o(out), weights

mha = MultiHeadAttention(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out, weights = mha(x, x, x)

print("=== Multi-Head Attention Forward Pass ===")
print(f"Input Tensor Shape   : {x.shape}")
print(f"Output Tensor Shape  : {out.shape}")
print(f"Attention Map Shape  : {weights.shape} [batch, heads, seq, seq]")